<a href="https://colab.research.google.com/github/perezbrotonsluis/movie-tv-recommender/blob/main/notebooks/02_preprocesamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Objetivo del preprocesamiento

El propósito fundamental de este notebook es preparar y refinar los datos de películas y series, sentando las bases para un sistema de recomendación robusto. Para lograrlo, la metodología se ha centrado en tres áreas clave: primero, consolidar toda la información relevante de títulos y créditos en un único conjunto de datos coherente; segundo, limpiar y normalizar dichos datos, eliminando duplicados, gestionando valores nulos y estandarizando formatos para facilitar análisis posteriores. Finalmente, se ha creado una columna unificada denominada `metadata`. Esta columna agrupa de manera estratégica la descripción, los géneros y los nombres de los participantes (como actores y directores), convirtiéndose en la piedra angular sobre la cual se calculará la similitud entre contenidos para ofrecer recomendaciones precisas y útiles.

In [1]:
# Carga de los datasets

import pandas as pd

movie_tv_titles = "https://raw.githubusercontent.com/perezbrotonsluis/movie-tv-recommender/main/data/raw/movie-tv-titles.csv"
movie_tv_credits = "https://raw.githubusercontent.com/perezbrotonsluis/movie-tv-recommender/main/data/raw/movie-tv-credits.csv"
df_titles = pd.read_csv(movie_tv_titles)
df_credits = pd.read_csv(movie_tv_credits)

# Muestra las primeras 5 filas del DataFrame
print('\n----- Dataframe movie-tv-titles -----\n')
display(df_titles.head())
print('\n\n----- Dataframe movie-tv-credits -----\n')
display(df_credits.head())


----- Dataframe movie-tv-titles -----



,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,platform
0,ts20945,The Three Stooges,SHOW,The Three Stooges were an American vaudeville ...,1934,TV-PG,19,"['comedy', 'family', 'animation', 'action', 'f...",['US'],26.0,tt0850645,8.6,1092.0,15.424,7.6,AmazonPrime
1,tm19248,The General,MOVIE,"During America’s Civil War, Union spies steal ...",1926,NaN,78,"['action', 'drama', 'war', 'western', 'comedy'...",['US'],NaN,tt0017925,8.2,89766.0,8.647,8.0,AmazonPrime
2,tm82253,The Best Years of Our Lives,MOVIE,It's the hope that sustains the spirit of ever...,1946,NaN,171,"['romance', 'war', 'drama']",['US'],NaN,tt0036868,8.1,63026.0,8.435,7.8,AmazonPrime
3,tm83884,His Girl Friday,MOVIE,"Hildy, the journalist former wife of newspaper...",1940,NaN,92,"['comedy', 'drama', 'romance']",['US'],NaN,tt0032599,7.8,57835.0,11.270,7.4,AmazonPrime
4,tm56584,In a Lonely Place,MOVIE,An aspiring actress begins to suspect that her...,1950,NaN,94,"['thriller', 'drama', 'romance']",['US'],NaN,tt0042593,7.9,30924.0,8.273,7.6,AmazonPrime




----- Dataframe movie-tv-credits -----



,person_id,id,name,character,role,platform
0,59401,ts20945,Joe Besser,Joe,ACTOR,AmazonPrime
1,31460,ts20945,Moe Howard,Moe,ACTOR,AmazonPrime
2,31461,ts20945,Larry Fine,Larry,ACTOR,AmazonPrime
3,21174,tm19248,Buster Keaton,Johnny Gray,ACTOR,AmazonPrime
4,28713,tm19248,Marion Mack,Annabelle Lee,ACTOR,AmazonPrime


##Definición formal del dataset final

Preparación del Dataset y Selección de Atributos

El objetivo de esta fase es definir con qué información va a trabajar realmente nuestro modelo. No todos los datos son útiles para calcular la similitud entre películas, por lo que hemos partido del dataset unificado para realizar una limpieza y selección estratégica de columnas.

Columnas conservadas:

- id y title: Fundamentales para la identificación única de cada obra y la visualización final de los resultados.

- description: Es el núcleo del modelo. Contiene la carga semántica principal que el TF-IDF transformará en vectores para encontrar tramas similares.

- genres: Clave para asegurar la coherencia temática. Al incluir los géneros, ayudamos al modelo a no mezclar categorías opuestas, aunque las descripciones compartan palabras comunes.

- name (actores y directores): Se ha decidido mantener esta columna porque puede ayudar a predictor de gustos en base a quien partidicpa en una película. Muchos usuarios buscan películas por un director concreto o un reparto recurrente.

Columnas descartadas: Se han eliminado metadatos irrelevantes para la recomendación textual, como fechas de estreno, países de producción o detalles técnicos de distribución. Mantener estas columnas solo añadiría "ruido" al modelo y aumentaría el coste computacional sin mejorar la calidad de las sugerencias.


In [2]:
# Seleccionar solo las columnas deseadas de los dataframes
df_titles_filtered = df_titles[['id', 'title', 'description', 'genres']]
df_credits_filtered = df_credits[['id', 'name']]

# Muestra las primeras 5 filas del DataFrame
print('\n----- Dataframe movie-tv-titles -----\n')
display(df_titles_filtered.head())
print('\n\n----- Dataframe movie-tv-credits -----\n')
display(df_credits_filtered.head())


----- Dataframe movie-tv-titles -----



,id,title,description,genres
0,ts20945,The Three Stooges,The Three Stooges were an American vaudeville ...,"['comedy', 'family', 'animation', 'action', 'f..."
1,tm19248,The General,"During America’s Civil War, Union spies steal ...","['action', 'drama', 'war', 'western', 'comedy'..."
2,tm82253,The Best Years of Our Lives,It's the hope that sustains the spirit of ever...,"['romance', 'war', 'drama']"
3,tm83884,His Girl Friday,"Hildy, the journalist former wife of newspaper...","['comedy', 'drama', 'romance']"
4,tm56584,In a Lonely Place,An aspiring actress begins to suspect that her...,"['thriller', 'drama', 'romance']"




----- Dataframe movie-tv-credits -----



,id,name
0,ts20945,Joe Besser
1,ts20945,Moe Howard
2,ts20945,Larry Fine
3,tm19248,Buster Keaton
4,tm19248,Marion Mack


###Unificación de ambos DataFrames

Antes de unir los DataFrames de títulos y créditos, es esencial preprocesar el DataFrame de créditos. Actualmente, cada actor o miembro del equipo tiene su propia fila para una misma película (mismo `id`), lo que generaría múltiples filas duplicadas de películas si se fusionaran directamente. Para evitar esto, se agruparán todos los nombres de participantes por `id` en una sola cadena de texto (`names_combined`). Así, cada película tendrá una única fila en el DataFrame final con todos sus créditos agrupados, facilitando una unificación limpia y correcta.

Además, como hay que preprocesar los nombres para que se vean como un único 'objeto', los vamos a preparar ya para que sea más fácil luego. También les pasamos un `.lower()` para que todo esté en minúsculas.

Para dejar la columna `names_combined` lista para el futuro recomendador TF-IDF, lo que hacemos es reemplazar los espacios de cada nombre por guiones bajos (`_`). Así, un nombre como 'Mark Hamill' se convierte en 'mark_hamill' y lo tratamos como una única palabra clave en el análisis de texto. De paso, también nos encargamos de los valores nulos para que no nos den problemas.

#### Formateo de los nombres para TF-IDF

In [3]:
df_credits_filtered = df_credits_filtered.copy()
df_credits_filtered['name'] = df_credits_filtered['name'].str.replace(' ', '_').str.lower()

display(df_credits_filtered.head())

,id,name
0,ts20945,joe_besser
1,ts20945,moe_howard
2,ts20945,larry_fine
3,tm19248,buster_keaton
4,tm19248,marion_mack


#### Agregando nombres de créditos por `id`

In [4]:
# Agrupar df_credits_filtered por 'id' y unir los nombres
df_credits_agg = df_credits_filtered.groupby('id')['name'].apply(lambda x: ', '.join(x)).reset_index()
df_credits_agg.rename(columns={'name': 'names_combined'}, inplace=True)

print('\n----- Dataframe de créditos agregado (df_credits_agg) -----\n')
display(df_credits_agg.head())

print(f"\nNúmero de filas en df_credits_filtered: {len(df_credits_filtered)}")
print(f"Número de filas en df_credits_agg (después de agrupar): {len(df_credits_agg)}")


----- Dataframe de créditos agregado (df_credits_agg) -----



,id,names_combined
0,tm1,"mark_hamill, harrison_ford, carrie_fisher, pet..."
1,tm10,"keanu_reeves, laurence_fishburne, carrie-anne_..."
2,tm100001,"john_wayne, barbara_sheldon, lloyd_whitlock, g..."
3,tm1000022,"chris_boike, nikki_stinson, tan_xiao, 张伟, leon..."
4,tm1000037,"luna_wedler, jannis_niewöhner, milan_peschel, ..."



Número de filas en df_credits_filtered: 294841
Número de filas en df_credits_agg (después de agrupar): 18686


#### Unificación Final

Con la preparación de `df_credits_agg` ya completada, donde cada `id` de película o serie ahora corresponde a una única fila que agrupa todos sus créditos (`names_combined`), el proceso de unificación con `df_titles_filtered` se puede llevar a cabo de manera eficiente garantizando que cada entrada sea única y contenga todos los datos necesarios para la construcción del recomendador sin generar filas duplicadas.

In [5]:
# Volver a realizar el merge con el DataFrame de créditos agregado
df = pd.merge(df_titles_filtered, df_credits_agg, on='id', how='left')

print('\n----- Dataframe unificado.shape() -----\n')
print(df.shape)

print('\n----- Dataframe unificado final (df_merged_final) -----\n')
display(df.head())


----- Dataframe unificado.shape() -----

(20550, 5)

----- Dataframe unificado final (df_merged_final) -----



,id,title,description,genres,names_combined
0,ts20945,The Three Stooges,The Three Stooges were an American vaudeville ...,"['comedy', 'family', 'animation', 'action', 'f...","joe_besser, moe_howard, larry_fine"
1,tm19248,The General,"During America’s Civil War, Union spies steal ...","['action', 'drama', 'war', 'western', 'comedy'...","buster_keaton, marion_mack, glen_cavender, jim..."
2,tm82253,The Best Years of Our Lives,It's the hope that sustains the spirit of ever...,"['romance', 'war', 'drama']","myrna_loy, fredric_march, dana_andrews, teresa..."
3,tm83884,His Girl Friday,"Hildy, the journalist former wife of newspaper...","['comedy', 'drama', 'romance']","cary_grant, rosalind_russell, ralph_bellamy, g..."
4,tm56584,In a Lonely Place,An aspiring actress begins to suspect that her...,"['thriller', 'drama', 'romance']","humphrey_bogart, gloria_grahame, frank_lovejoy..."


## Preprocesamiento Estructural

En esta etapa, el objetivo principal es garantizar que el dataset sea usable, no “perfecto”. Se realizaron acciones clave como la eliminación de duplicados, el manejo de valores nulos y el reseteo de índices. Estas operaciones son fundamentales para asegurar la consistencia mínima del DataFrame antes de la construcción de la columna `metadata` y la aplicación de algoritmos de recomendación.

In [6]:
# Mostrar la cantidad de valores duplicados
print(f"\nDuplicated():")
display(df.duplicated().sum())

# Mostrar la cantidad de valores nulos por columna
print(f"\nIsnull().sum():")
display(df.isnull().sum())

# Mostrar la cantidad de valores nulos por columna (%)
print(f"\nIsnull().mean()*100:")
display(df.isnull().mean()*100)


Duplicated():


np.int64(132)


Isnull().sum():


,0
id,0
title,1
description,155
genres,0
names_combined,1642



Isnull().mean()*100:


,0
id,0.000000
title,0.004866
description,0.754258
genres,0.000000
names_combined,7.990268


Tras revisar los datos de duplicados y valores nulos, se tomaron las siguientes decisiones fundamentales para la limpieza del DataFrame:

1.  **Duplicados**: Se detectaron 132 filas duplicadas. La decisión fue eliminar todas estas entradas, ya que representan redundancia y podrían sesgar el análisis o el modelo de recomendación.

2.  **Valores Nulos en `title`**: La presencia de un `title` nulo hace que un registro sea inviable para un recomendador, puesto que no se podría identificar qué se está recomendando. Por ello, todas las filas con `title` nulo fueron eliminadas.

3.  **Valores Nulos en `description`**: Aunque el porcentaje de nulos en la columna `description` era muy bajo (inferior al 1%), se decidió eliminar estas filas. La `description` es la fuente más rica de información para el recomendador basado en contenido, y la ausencia de la misma empobrecería significativamente la calidad de la recomendación para esos títulos.

4.  **Valores Nulos en `names_combined`**: Esta columna presentaba un porcentaje de nulos más elevado (casi un 8%). Sin embargo, se consideró que no era una razón suficiente para eliminar filas completas. En su lugar, los valores nulos en `names_combined` fueron rellenados con una cadena vacía. La lógica detrás de esta decisión es que, si bien los nombres son importantes, la `description` tiene un peso mayor en la `metadata` y el recomendador aún puede funcionar bien con esta información faltante, sin perder un 8% del dataset.

In [7]:
# 1. Eliminar duplicados
initial_rows = len(df)
df.drop_duplicates(inplace=True)
df.drop_duplicates(subset=['id'], inplace=True)
df.drop_duplicates(subset=['title'], inplace=True)
print(f"Se eliminaron {initial_rows - len(df)} filas duplicadas.")

# 2. Eliminar filas donde 'title' o 'description' sean nulos
new_initial_rows = len(df)
df.dropna(subset=['title'], inplace=True)
df.dropna(subset=['description'], inplace=True)
print(f"Se eliminaron {new_initial_rows - len(df)} filas con 'title' o 'description' nulos.")

# 3. Rellenar valores nulos en 'names_combined' con una cadena vacía
df['names_combined'] = df['names_combined'].fillna('')
print("Valores nulos en 'names_combined' rellenados con cadena vacía.")

# 4. Resetear el índice
df.reset_index(drop=True, inplace=True)
print("Índice del DataFrame reseteado.")

# Mostrar el estado final del DataFrame después de la limpieza
print(f"\nNúmero total de filas anres de la limpieza: {initial_rows}")
print(f"\nNúmero total de filas después de la limpieza: {len(df)}")
print(f"\nValores nulos después de la limpieza:\n{df.isnull().sum()}")
display(df.head())

Se eliminaron 738 filas duplicadas.
Se eliminaron 151 filas con 'title' o 'description' nulos.
Valores nulos en 'names_combined' rellenados con cadena vacía.
Índice del DataFrame reseteado.

Número total de filas anres de la limpieza: 20550

Número total de filas después de la limpieza: 19661

Valores nulos después de la limpieza:
id                0
title             0
description       0
genres            0
names_combined    0
dtype: int64


,id,title,description,genres,names_combined
0,ts20945,The Three Stooges,The Three Stooges were an American vaudeville ...,"['comedy', 'family', 'animation', 'action', 'f...","joe_besser, moe_howard, larry_fine"
1,tm19248,The General,"During America’s Civil War, Union spies steal ...","['action', 'drama', 'war', 'western', 'comedy'...","buster_keaton, marion_mack, glen_cavender, jim..."
2,tm82253,The Best Years of Our Lives,It's the hope that sustains the spirit of ever...,"['romance', 'war', 'drama']","myrna_loy, fredric_march, dana_andrews, teresa..."
3,tm83884,His Girl Friday,"Hildy, the journalist former wife of newspaper...","['comedy', 'drama', 'romance']","cary_grant, rosalind_russell, ralph_bellamy, g..."
4,tm56584,In a Lonely Place,An aspiring actress begins to suspect that her...,"['thriller', 'drama', 'romance']","humphrey_bogart, gloria_grahame, frank_lovejoy..."


## Construcción de la Columna `metadata`

En esta fase crucial, se procede a la creación de una nueva columna denominada `metadata` (o `combined_features`). Esta columna es la pieza central para nuestro sistema de recomendación basado en contenido, ya que agrupa la información textual más relevante de cada título en un único campo. Concretamente, se concatenan los siguientes elementos:

-   La **`description`** del contenido.
-   Los **`genres`** asociados.
-   Los **`names_combined`**, que incluyen tanto a los actores como a los directores.

El objetivo es tener un "documento" unificado por cada película o serie, que será posteriormente procesado por técnicas como TF-IDF para medir la similitud entre contenidos. Conceptualmente, el resultado sería algo así:

`metadata = description + genres + names_combined`

### Preparación de la Columna `genres` para `metadata`

Antes de concatenar la columna `genres` en la `metadata` final, se realiza un paso de preprocesamiento específico para asegurar que su formato sea el adecuado. Inicialmente, la columna `genres` puede contener sus valores como cadenas de texto que representan listas de Python (por ejemplo, `"['Action', 'Comedy']"`). Si se concatenara directamente en este formato, no se obtendría el resultado deseado para el análisis de texto.

Por ello, en el código se realizan dos acciones:

1.  **`df['genres'].apply(ast.literal_eval)`**: Esta línea convierte la cadena de texto de la columna `genres` en una lista real de Python. Esto es fundamental para poder manipular sus elementos de manera programática.
2.  **`df['genres'].apply(lambda x: ', '.join(x))`**: Una vez que `genres` es una lista de Python, esta línea une todos los elementos de esa lista en una única cadena de texto, separándolos por una coma y un espacio (ej., `'Action, Comedy'`).

Este proceso garantiza que, al formar la columna `metadata`, los géneros se integren como una cadena de texto única y coherente, facilitando así su posterior tokenización y análisis por parte del recomendador TF-IDF.

In [8]:
import ast

# 1. Convertir el texto de "lista" a una lista de verdad de Python
df['genres'] = df['genres'].apply(ast.literal_eval)

# 2. Unir los elementos con una coma y un espacio
df['genres'] = df['genres'].apply(lambda x: ', '.join(x))

# Ver el resultado
display(df['genres'].head())

,genres
0,"comedy, family, animation, action, fantasy, ho..."
1,"action, drama, war, western, comedy, european"
2,"romance, war, drama"
3,"comedy, drama, romance"
4,"thriller, drama, romance"


In [9]:
# Concatenar las columnas 'description', 'genres' y 'names_combined' en una nueva columna 'metadata'
# Rellenamos cualquier valor nulo con una cadena vacía antes de concatenar para evitar errores y asegurar un string.
df['metadata'] = df['description'] + ' ' + df['genres'] + ' ' + df['names_combined']

# Mostrar las primeras filas con la nueva columna 'metadata' para verificar el resultado
print('\nDataFrame con la nueva columna metadata:\n')
display(df.head())

print('\nMetadata de la primera pelicula\n')
display(df['metadata'].iloc[0])


DataFrame con la nueva columna metadata:



,id,title,description,genres,names_combined,metadata
0,ts20945,The Three Stooges,The Three Stooges were an American vaudeville ...,"comedy, family, animation, action, fantasy, ho...","joe_besser, moe_howard, larry_fine",The Three Stooges were an American vaudeville ...
1,tm19248,The General,"During America’s Civil War, Union spies steal ...","action, drama, war, western, comedy, european","buster_keaton, marion_mack, glen_cavender, jim...","During America’s Civil War, Union spies steal ..."
2,tm82253,The Best Years of Our Lives,It's the hope that sustains the spirit of ever...,"romance, war, drama","myrna_loy, fredric_march, dana_andrews, teresa...",It's the hope that sustains the spirit of ever...
3,tm83884,His Girl Friday,"Hildy, the journalist former wife of newspaper...","comedy, drama, romance","cary_grant, rosalind_russell, ralph_bellamy, g...","Hildy, the journalist former wife of newspaper..."
4,tm56584,In a Lonely Place,An aspiring actress begins to suspect that her...,"thriller, drama, romance","humphrey_bogart, gloria_grahame, frank_lovejoy...",An aspiring actress begins to suspect that her...



Metadata de la primera pelicula



"The Three Stooges were an American vaudeville and comedy team active from 1922 until 1970, best known for their 190 short subject films by Columbia Pictures that have been regularly airing on television since 1958. Their hallmark was physical farce and slapstick. In films, the stooges were commonly known by their actual first names. There were a total of six stooges over the act's run (with only three active at any given time), but Moe Howard and Larry Fine were the mainstays throughout the ensemble's nearly fifty-year run. comedy, family, animation, action, fantasy, horror joe_besser, moe_howard, larry_fine"

## Preprocesamiento del Texto en `metadata`

Para optimizar la columna `metadata` de cara al modelo TF-IDF, se ha llevado a cabo un preprocesamiento del texto. Este proceso incluye convertir todo a minúsculas, eliminar caracteres especiales y puntuación (manteniendo los guiones bajos para nombres compuestos) y normalizar los espacios, asegurando así un formato estándar y consistente para el análisis.

In [10]:
import re

# 1. Convertir todo a minúsculas
df['metadata'] = df['metadata'].str.lower()

# 2. Eliminar caracteres especiales y puntuación, manteniendo guiones bajos y espacios
# Cualquier carácter que no sea una letra (a-z), un número (0-9), un guion bajo (_) o un espacio se reemplazará por un espacio.
df['metadata'] = df['metadata'].apply(lambda x: re.sub(r'[^a-z0-9_ ]', ' ', x))

# 3. Eliminar espacios extra (múltiples espacios a uno solo, y espacios al inicio/final)
df['metadata'] = df['metadata'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

print("Metadata después del preprocesamiento:")
display(df[['id', 'title', 'metadata']].head())

print('\nMetadata de la primera pelicula\n')
display(df['metadata'].iloc[0])

Metadata después del preprocesamiento:


,id,title,metadata
0,ts20945,The Three Stooges,the three stooges were an american vaudeville ...
1,tm19248,The General,during america s civil war union spies steal e...
2,tm82253,The Best Years of Our Lives,it s the hope that sustains the spirit of ever...
3,tm83884,His Girl Friday,hildy the journalist former wife of newspaper ...
4,tm56584,In a Lonely Place,an aspiring actress begins to suspect that her...



Metadata de la primera pelicula



'the three stooges were an american vaudeville and comedy team active from 1922 until 1970 best known for their 190 short subject films by columbia pictures that have been regularly airing on television since 1958 their hallmark was physical farce and slapstick in films the stooges were commonly known by their actual first names there were a total of six stooges over the act s run with only three active at any given time but moe howard and larry fine were the mainstays throughout the ensemble s nearly fifty year run comedy family animation action fantasy horror joe_besser moe_howard larry_fine'

In [11]:
df.to_csv('movie_tv_clean.csv', index=False)